In [ ]:
import json
import math
import pandas as pd
import numpy as np
from shapely.geometry import shape, Point
import gamspy as gp

# ----------------------------------------------------
# 1. Load Data & Geometries
# ----------------------------------------------------
# Load district population
df_pop = pd.read_excel(f"Data\جمعیت مناطق-1405-06-25.xlsx")
pop_dict = {
    1: 80807, 2: 76733, 3: 111013, 4: 147139, 5: 132984,
    6: 112896, 7: 206588, 8: 243563, 9: 78271, 10: 200702,
    11: 58334, 12: 155413, 13: 164413, 14: 160354, 15: 138247
}

# Parse 15 district polygons and calculate centroids
with open(r"Data\isfahan_raw.geojson", "r", encoding="utf-8") as f:
    geojson_data = json.load(f)

persian_to_eng = {
    '۱': 1, '۲': 2, '۳': 3, '۴': 4, '۵': 5, '۶': 6, '۷': 7, '۸': 8,
    '۹': 9, '۱۰': 10, '۱۱': 11, '۱۲': 12, '۱۳': 13, '۱۴': 14, '۱۵': 15
}

districts = []
for feat in geojson_data["features"]:
    geom_type = feat.get("geometry", {}).get("type")
    props = feat.get("properties", {})
    name = props.get("name", "")
    admin_level = str(props.get("admin_level", ""))

    if geom_type not in ["Polygon", "MultiPolygon"] or admin_level != "9" or "خمینی" in name:
        continue

    d_num = None
    for p_num, val in persian_to_eng.items():
        if f"منطقه {p_num}" == name.strip() or f"منطقه {p_num} " in name:
            d_num = val
            break
    if d_num:
        poly = shape(feat["geometry"])
        districts.append({
            "id": f"D{d_num}",
            "d_num": d_num,
            "name": name,
            "lat": poly.centroid.y,
            "lon": poly.centroid.x,
            "geom": poly,
            "population": pop_dict[d_num]
        })

df_districts = pd.DataFrame(districts).sort_values("d_num").reset_index(drop=True)

In [ ]:
refined_targets = np.load(r"Data\refined_targets.npy", allow_pickle=True)
target_counts = {d["id"]: 0 for d in districts}
for t in refined_targets:
    pt = Point(t["lon"], t["lat"])
    for d in districts:
        if d["geom"].contains(pt):
            target_counts[d["id"]] += 1
            break
refined_targets

In [5]:
# ----------------------------------------------------
# 3. Distance & Reachability Matrices (Haversine km)
# ----------------------------------------------------
stations = [
    {"name": "Qods (Malek Shahr)", "lat": 32.71472, "lon": 51.60472},
    {"name": "Baharestan",          "lat": 32.71528, "lon": 51.62528},
    {"name": "Golestan",            "lat": 32.71528, "lon": 51.62722},
    {"name": "Shahid Mofateh",      "lat": 32.71667, "lon": 51.64528},
    {"name": "Shahid Alikhani",     "lat": 32.71611, "lon": 51.66083},
    {"name": "Jaber",               "lat": 32.70889, "lon": 51.66722},
    {"name": "Kaveh",               "lat": 32.69806, "lon": 51.67417},
    {"name": "Shahid Chamran",      "lat": 32.68750, "lon": 51.67583},
    {"name": "Shahid Bahonar",      "lat": 32.67889, "lon": 51.67556},
    {"name": "Shohada",             "lat": 32.67222, "lon": 51.67250},
    {"name": "Takhti",              "lat": 32.66537, "lon": 51.67044},
    {"name": "Emam Hossein",        "lat": 32.65785, "lon": 51.66956},
    {"name": "Enqelab",             "lat": 32.64944, "lon": 51.66840},
    {"name": "Si-o-se Pol",         "lat": 32.63929, "lon": 51.66670},
    {"name": "Shari'ati",           "lat": 32.62830, "lon": 51.66550},
    {"name": "Azadi",               "lat": 32.62236, "lon": 51.66483},
    {"name": "Daneshgah-e Esfahan", "lat": 32.61444, "lon": 51.66371},
    {"name": "Kargar",              "lat": 32.60694, "lon": 51.66376},
    {"name": "Kuy-e Emam",          "lat": 32.59823, "lon": 51.66861},
    {"name": "Defa'-e Moqaddas (Soffeh)", "lat": 32.58983, "lon": 51.67031},
]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    return 2 * R * math.asin(math.sqrt(a))

d_max_km = 4.5  # Acceptable evacuation radius between district centroids

A_data = []
for _, row1 in df_districts.iterrows():
    for _, row2 in df_districts.iterrows():
        dist = haversine(row1["lat"], row1["lon"], row2["lat"], row2["lon"])
        if dist <= d_max_km:
            A_data.append((row1["id"], row2["id"], 1))

B_data = []
for _, row in df_districts.iterrows():
    for idx, s in enumerate(stations):
        sub_id = f"S{idx+1}"
        dist = haversine(row["lat"], row["lon"], s["lat"], s["lon"])
        if dist <= d_max_km:
            B_data.append((row["id"], sub_id, 1))

In [ ]:
# ----------------------------------------------------
# Distance-decayed strategic risk exposure per district
# ----------------------------------------------------
risk_scores = {d["id"]: 0.0 for d in districts}
for t in refined_targets:
    t_lat, t_lon = t["lat"], t["lon"]
    for d in districts:
        centroid = d["geom"].centroid
        dist = haversine(centroid.y, centroid.x, t_lat, t_lon)
        risk_scores[d["id"]] += 1 / (1 + dist)

max_risk = max(risk_scores.values()) if risk_scores else 1.0
risk_normalized = {k: v / max_risk for k, v in risk_scores.items()}

In [ ]:
# ----------------------------------------------------
# 4. GAMSPy Mathematical Model
# ----------------------------------------------------
m = gp.Container()

# Sets
i = gp.Set(m, name="i", records=df_districts["id"].tolist())
alias_i = gp.Alias(m, name="ip", alias_with=i)
k = gp.Set(m, name="k", records=[f"S{idx+1}" for idx in range(len(stations))])

# Capacity specifications
C_val = 20000         # Capacity per standard modular shelter unit
CapSub_val = 15000    # Evacuation platform capacity per subway station

# Parameters
P = gp.Parameter(m, name="P", domain=[i], records=df_districts[["id", "population"]])
Risk = gp.Parameter(m, name="Risk", domain=[i],
                     records=pd.DataFrame(list(risk_normalized.items()), columns=["id", "score"]))
A = gp.Parameter(m, name="A", domain=[i, alias_i], records=pd.DataFrame(A_data, columns=["i", "ip", "value"]))
B = gp.Parameter(m, name="B", domain=[i, k], records=pd.DataFrame(B_data, columns=["i", "k", "value"]))

# Subway platform capacity parameter
CapSub_records = pd.DataFrame([{"k": f"S{idx+1}", "capacity": CapSub_val} for idx in range(len(stations))])
CapSub = gp.Parameter(m, name="CapSub", domain=[k], records=CapSub_records)

# Capacity specifications
C_val = 15000         # Capacity per standard modular shelter unit
CapSub_val = 10000    # Evacuation platform capacity per subway station
CapSub = gp.Parameter(m, name="CapSub", domain=[k], records=[[f"S{idx+1}", CapSub_val] for idx in range(len(stations))])

# Decision Variables
Y = gp.Variable(m, name="Y", domain=[i], type="Integer")  # Number of shelter units in district i
x = gp.Variable(m, name="x", domain=[i, alias_i], type="Positive")  # Evacuees from district i to shelter i'
w = gp.Variable(m, name="w", domain=[i, k], type="Positive")        # Evacuees from district i to subway k

# Equations
demand_cov = gp.Equation(m, name="demand_cov", domain=[i])
demand_cov[i] = gp.Sum(alias_i, x[i, alias_i]) + gp.Sum(k, w[i, k]) == P[i]

reach_shelter = gp.Equation(m, name="reach_shelter", domain=[i, alias_i])
reach_shelter[i, alias_i] = x[i, alias_i] <= P[i] * A[i, alias_i]

reach_subway = gp.Equation(m, name="reach_subway", domain=[i, k])
reach_subway[i, k] = w[i, k] <= P[i] * B[i, k]

cap_shelter = gp.Equation(m, name="cap_shelter", domain=[alias_i])
cap_shelter[alias_i] = gp.Sum(i, x[i, alias_i]) <= C_val * Y[alias_i]

cap_subway = gp.Equation(m, name="cap_subway", domain=[k])
cap_subway[k] = gp.Sum(i, w[i, k]) <= CapSub[k]

# Target risk scaling: Ensures districts with higher target density establish baseline shelter units
risk_coverage_fraction = 0.25
target_risk_rule = gp.Equation(m, name="target_risk_rule", domain=[i])
target_risk_rule[i] = Y[i] * C_val >= risk_coverage_fraction * Risk[i] * P[i]

# Objective: Minimize total shelter units constructed
obj = gp.Sum(i, Y[i])

shelter_model = gp.Model(
    m,
    name="Isfahan_Shelter_Location",
    equations=m.getEquations(),
    problem="MIP",
    sense=gp.Sense.MIN,
    objective=obj
)

shelter_model.solve()

,Solver Status,Model Status,Objective,Num of Equations,Num of Variables,Model Type,Solver,Solver Time
0,NormalCompletion,OptimalGlobal,128.0,591.0,541.0,MIP,CPLEX,0.127


In [19]:
# ----------------------------------------------------
# 5. Extract Results
# ----------------------------------------------------
print("--- Optimization Results ---")
print(f"Objective Status: {shelter_model.status}")
print(f"Total Shelter Units Required: {int(shelter_model.objective_value)}")
results_df = Y.records[["i", "level"]].rename(columns={"i": "District", "level": "Shelter_Units"})
print(results_df[results_df["Shelter_Units"] > 0])

--- Optimization Results ---
Objective Status: ModelStatus.OptimalGlobal
Total Shelter Units Required: 128
   District  Shelter_Units
0        D1            7.0
1        D2            6.0
3        D4           10.0
4        D5            7.0
5        D6            8.0
8        D9            5.0
9       D10           14.0
10      D11           17.0
11      D12           11.0
12      D13           11.0
13      D14           22.0
14      D15           10.0


In [11]:
shelter_model.toLatex(path="latex", generate_pdf=False)

LaTeX (.tex) file has been generated under latex\Isfahan_Shelter_Location.tex
